# Capacity Semantic Model Inventory

This notebook extracts semantic model information from **all workspaces on a capacity** and loads the results into Fabric lakehouse delta tables.

### Prerequisites
- A Fabric lakehouse must be attached to this notebook
- The user must have admin or capacity-level permissions to enumerate workspaces
- The `semantic-link-labs` package must be installed

### Delta Tables Created
| Table Name | Description |
|---|---|
| `capacity_workspaces` | All workspaces on the capacity |
| `capacity_semantic_models` | All semantic models across those workspaces |
| `capacity_model_tables` | Tables in each semantic model |
| `capacity_model_columns` | Columns in each semantic model |
| `capacity_model_measures` | Measures in each semantic model |
| `capacity_model_relationships` | Relationships in each semantic model |
| `capacity_model_partitions` | Partitions in each semantic model |
| `capacity_model_bpa` | Best Practice Analyzer results |

In [ ]:
%pip install semantic-link-labs -q

## Configuration

Set your capacity name or ID below. All workspaces assigned to this capacity will be scanned.

In [ ]:
capacity_name = "Your Capacity Name"  # Set to capacity name or UUID

## Imports

In [ ]:
import sempy.fabric as fabric
import sempy_labs as labs
from sempy_labs.admin import list_workspaces, list_datasets
from sempy_labs import save_as_delta_table
from sempy_labs.lakehouse import lakehouse_attached
import pandas as pd
import datetime

## Validate Lakehouse Attachment

In [ ]:
if not lakehouse_attached():
    raise RuntimeError(
        "A lakehouse must be attached to this notebook before running. "
        "Please attach a lakehouse and re-run."
    )

## Step 1 — List All Workspaces on the Capacity

In [ ]:
df_workspaces = list_workspaces(capacity=capacity_name)
df_workspaces = df_workspaces[df_workspaces["State"] == "Active"]

print(f"Found {len(df_workspaces)} active workspaces on capacity '{capacity_name}'.")
df_workspaces.head(10)

## Step 2 — Save Workspace Inventory to Lakehouse

In [ ]:
now = datetime.datetime.now()
df_ws_export = df_workspaces.copy()
df_ws_export["Timestamp"] = now

save_as_delta_table(
    dataframe=df_ws_export,
    delta_table_name="capacity_workspaces",
    write_mode="overwrite",
)
print(f"Saved {len(df_ws_export)} workspaces to 'capacity_workspaces' table.")

## Step 3 — Discover All Semantic Models Across Workspaces

Iterates through each workspace and lists all semantic models (datasets).

In [ ]:
all_models = []

for _, ws in df_workspaces.iterrows():
    ws_name = ws["Name"]
    ws_id = ws["Id"]
    try:
        df_datasets = fabric.list_datasets(workspace=ws_id, mode="rest")
        if not df_datasets.empty:
            df_datasets["Workspace Name"] = ws_name
            df_datasets["Workspace Id"] = ws_id
            all_models.append(df_datasets)
    except Exception as e:
        print(f"  Skipping workspace '{ws_name}': {e}")

if all_models:
    df_all_models = pd.concat(all_models, ignore_index=True)
else:
    df_all_models = pd.DataFrame()

print(f"Found {len(df_all_models)} semantic models across {len(df_workspaces)} workspaces.")
df_all_models.head(10)

## Step 4 — Save Semantic Model Inventory to Lakehouse

In [ ]:
if not df_all_models.empty:
    df_models_export = df_all_models.copy()
    df_models_export["Timestamp"] = now

    save_as_delta_table(
        dataframe=df_models_export,
        delta_table_name="capacity_semantic_models",
        write_mode="overwrite",
        merge_schema=True,
    )
    print(f"Saved {len(df_models_export)} models to 'capacity_semantic_models' table.")
else:
    print("No semantic models found.")

## Step 5 — Extract Detailed Model Metadata

For each semantic model, extract tables, columns, measures, relationships, and partitions.

**Note**: This step iterates through every model and may take some time for large environments.

In [ ]:
all_tables = []
all_columns = []
all_measures = []
all_relationships = []
all_partitions = []
errors = []

total = len(df_all_models)

for idx, (_, row) in enumerate(df_all_models.iterrows(), 1):
    dataset_name = row["Dataset Name"]
    dataset_id = row["Dataset Id"]
    ws_name = row["Workspace Name"]
    ws_id = row["Workspace Id"]

    print(f"[{idx}/{total}] Processing '{dataset_name}' in '{ws_name}'...")

    prefix = {
        "Workspace Name": ws_name,
        "Workspace Id": ws_id,
        "Dataset Name": dataset_name,
        "Dataset Id": dataset_id,
    }

    # Tables
    try:
        df_t = labs.list_tables(dataset=dataset_id, workspace=ws_id, extended=True)
        if not df_t.empty:
            for k, v in prefix.items():
                df_t.insert(0, k, v) if k not in df_t.columns else None
            all_tables.append(df_t)
    except Exception as e:
        errors.append({"Model": dataset_name, "Workspace": ws_name, "Object": "Tables", "Error": str(e)})

    # Columns
    try:
        df_c = labs.list_columns(dataset=dataset_id, workspace=ws_id)
        if not df_c.empty:
            for k, v in prefix.items():
                df_c.insert(0, k, v) if k not in df_c.columns else None
            all_columns.append(df_c)
    except Exception as e:
        errors.append({"Model": dataset_name, "Workspace": ws_name, "Object": "Columns", "Error": str(e)})

    # Measures
    try:
        df_m = fabric.list_measures(dataset=dataset_id, workspace=ws_id)
        if not df_m.empty:
            for k, v in prefix.items():
                df_m.insert(0, k, v) if k not in df_m.columns else None
            all_measures.append(df_m)
    except Exception as e:
        errors.append({"Model": dataset_name, "Workspace": ws_name, "Object": "Measures", "Error": str(e)})

    # Relationships
    try:
        df_r = labs.list_relationships(dataset=dataset_id, workspace=ws_id, extended=True)
        if not df_r.empty:
            for k, v in prefix.items():
                df_r.insert(0, k, v) if k not in df_r.columns else None
            all_relationships.append(df_r)
    except Exception as e:
        errors.append({"Model": dataset_name, "Workspace": ws_name, "Object": "Relationships", "Error": str(e)})

    # Partitions
    try:
        df_p = fabric.list_partitions(dataset=dataset_id, workspace=ws_id)
        if not df_p.empty:
            for k, v in prefix.items():
                df_p.insert(0, k, v) if k not in df_p.columns else None
            all_partitions.append(df_p)
    except Exception as e:
        errors.append({"Model": dataset_name, "Workspace": ws_name, "Object": "Partitions", "Error": str(e)})

print(f"\nDone. Encountered {len(errors)} errors.")
if errors:
    df_errors = pd.DataFrame(errors)
    display(df_errors)

## Step 6 — Save Model Metadata to Lakehouse Tables

In [ ]:
def save_collected_data(frames, table_name):
    """Concatenate collected DataFrames and save to a delta table."""
    if frames:
        df = pd.concat(frames, ignore_index=True)
        df["Timestamp"] = now
        save_as_delta_table(
            dataframe=df,
            delta_table_name=table_name,
            write_mode="overwrite",
            merge_schema=True,
        )
        print(f"  Saved {len(df)} rows to '{table_name}'.")
    else:
        print(f"  No data for '{table_name}'.")


save_collected_data(all_tables, "capacity_model_tables")
save_collected_data(all_columns, "capacity_model_columns")
save_collected_data(all_measures, "capacity_model_measures")
save_collected_data(all_relationships, "capacity_model_relationships")
save_collected_data(all_partitions, "capacity_model_partitions")

print("\nAll model metadata saved to lakehouse.")

## Step 7 — Run Best Practice Analyzer Across All Models

Runs the Model BPA for each semantic model and collects the results into a single table.

**Note**: This step uses TOM connections and may take significant time for large environments. You can skip this cell if BPA results are not needed.

In [ ]:
all_bpa = []
bpa_errors = []

for idx, (_, row) in enumerate(df_all_models.iterrows(), 1):
    dataset_name = row["Dataset Name"]
    dataset_id = row["Dataset Id"]
    ws_name = row["Workspace Name"]
    ws_id = row["Workspace Id"]

    print(f"[{idx}/{total}] BPA for '{dataset_name}' in '{ws_name}'...")

    try:
        df_bpa = labs.run_model_bpa(
            dataset=dataset_id,
            workspace=ws_id,
            return_dataframe=True,
        )
        if df_bpa is not None and not df_bpa.empty:
            df_bpa["Workspace Name"] = ws_name
            df_bpa["Workspace Id"] = ws_id
            df_bpa["Dataset Name"] = dataset_name
            df_bpa["Dataset Id"] = dataset_id
            all_bpa.append(df_bpa)
    except Exception as e:
        bpa_errors.append({"Model": dataset_name, "Workspace": ws_name, "Error": str(e)})

print(f"\nBPA complete. Encountered {len(bpa_errors)} errors.")
if bpa_errors:
    display(pd.DataFrame(bpa_errors))

In [ ]:
save_collected_data(all_bpa, "capacity_model_bpa")
print("BPA results saved to lakehouse.")

## Step 8 — Run Vertipaq Analyzer Across All Models (Optional)

Uses the built-in `export='table'` mode which automatically creates delta tables:
- `vertipaqanalyzer_model`
- `vertipaqanalyzer_tables`
- `vertipaqanalyzer_columns`
- `vertipaqanalyzer_partitions`
- `vertipaqanalyzer_relationships`
- `vertipaqanalyzer_hierarchies`

Each run appends data with workspace/model context columns and a `RunId`.

**Note**: Requires TOM connections. Can be slow for large models. Skip this cell if not needed.

In [ ]:
vertipaq_errors = []

for idx, (_, row) in enumerate(df_all_models.iterrows(), 1):
    dataset_name = row["Dataset Name"]
    dataset_id = row["Dataset Id"]
    ws_name = row["Workspace Name"]
    ws_id = row["Workspace Id"]

    print(f"[{idx}/{total}] Vertipaq Analyzer for '{dataset_name}' in '{ws_name}'...")

    try:
        labs.vertipaq_analyzer(
            dataset=dataset_id,
            workspace=ws_id,
            export="table",
        )
    except Exception as e:
        vertipaq_errors.append({"Model": dataset_name, "Workspace": ws_name, "Error": str(e)})

print(f"\nVertipaq Analyzer complete. Encountered {len(vertipaq_errors)} errors.")
if vertipaq_errors:
    display(pd.DataFrame(vertipaq_errors))

## Summary

Run the cell below to show a summary of all tables created.

In [ ]:
from sempy_labs.lakehouse import get_lakehouse_tables

df_lh_tables = get_lakehouse_tables()

target_tables = [
    "capacity_workspaces",
    "capacity_semantic_models",
    "capacity_model_tables",
    "capacity_model_columns",
    "capacity_model_measures",
    "capacity_model_relationships",
    "capacity_model_partitions",
    "capacity_model_bpa",
    "vertipaqanalyzer_model",
    "vertipaqanalyzer_tables",
    "vertipaqanalyzer_columns",
    "vertipaqanalyzer_partitions",
    "vertipaqanalyzer_relationships",
    "vertipaqanalyzer_hierarchies",
]

df_summary = df_lh_tables[df_lh_tables["Table Name"].isin(target_tables)]
print(f"Lakehouse tables created ({len(df_summary)}):")
display(df_summary)